# Notebook 16 – Feature Engineering Mini Challenge

The dataset used here is `data.csv` — a raw log of online retail transactions.

## Step 1: Understand the Dataset

First, load the raw data and inspect its shape, columns, and types — before deciding anything else.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
print("Shape:", df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


**Observations:**
- Each row is a single **line item** in a transaction (one product, one invoice), not one customer.
- Columns: `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`.
- There is **no ready-made target column** — this is raw transactional data, so a target has to be defined based on a real business question.
- `CustomerID` has missing values, and some `Quantity`/`UnitPrice` values are negative (likely returns or data errors).

In [3]:
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print("Cleaned shape:", df.shape)
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print("Unique customers:", df['CustomerID'].nunique())

Cleaned shape: (397884, 9)
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique customers: 4338


## Step 2: Identify the Target

**Business question:** *Will a customer make another purchase in the future?* This is a classic **customer retention / churn** problem — genuinely useful, since it lets a business focus retention efforts on at-risk customers.

To build this target **without leakage**, we split the timeline at a cutoff date:
- **Feature window:** all transactions *before* the cutoff — used to build customer features.
- **Target window:** all transactions *after* the cutoff — used only to define the label (did they come back or not).

This mirrors a real production setting, where "the future" genuinely hasn't happened yet when features are computed.

In [4]:
cutoff = pd.Timestamp('2011-09-01')
pre_cutoff = df[df['InvoiceDate'] < cutoff]   
post_cutoff = df[df['InvoiceDate'] >= cutoff] 
target_df = pre_cutoff[['CustomerID']].drop_duplicates()
target_df['WillPurchaseAgain'] = target_df['CustomerID'].isin(post_cutoff['CustomerID']).astype(int)
print(target_df['WillPurchaseAgain'].value_counts())
target_df.head()

WillPurchaseAgain
1    1952
0    1365
Name: count, dtype: int64


,CustomerID,WillPurchaseAgain
0,17850.0,0
9,13047.0,1
26,12583.0,1
46,13748.0,1
65,15100.0,0


## Step 3: Identify Existing (Raw) Features

The raw columns available before any engineering:

| Column | Type | Usefulness as-is |
|---|---|---|
| `InvoiceNo` | Identifier | Not predictive alone, but useful for counting orders |
| `StockCode` | Identifier | Too high-cardinality to use directly |
| `Description` | Text | Not usable directly without NLP processing |
| `Quantity` | Numeric | Useful, but only meaningful per-transaction, not per-customer |
| `InvoiceDate` | Datetime | Very useful once broken into derived features |
| `UnitPrice` | Numeric | Useful, but needs aggregation to the customer level |
| `CustomerID` | Identifier | Needed as a grouping key, not a model input |
| `Country` | Categorical | Usable directly with encoding |

None of these, on their own, describe a **customer** — they describe individual transaction lines. This is exactly why feature engineering is needed: we must aggregate transaction-level data up to customer-level features.

## Step 4: Identify Potentially Useful New Features

Based on the business question (will this customer return?), useful signals likely include:
- **How much** they've spent (monetary value)
- **How often** they buy (frequency)
- **How recently** they last bought (recency)
- **How long** they've been a customer (tenure)
- **How varied** their purchases are (product/month diversity)
- **Where** they're from (country)
- **When** they tend to shop (day-of-week / weekend pattern)

This is essentially an **RFM (Recency, Frequency, Monetary)** framework, extended with a few extra behavioral signals.

## Step 5: Create at Least 10 Meaningful Features

All features below are computed **only from `pre_cutoff` data**, to avoid leaking information from the target window.

In [8]:
customer_features = pre_cutoff.groupby('CustomerID').agg(
    TotalSpend=('TotalPrice', 'sum'),
    PurchaseCount=('InvoiceNo', 'nunique'),
    AvgOrderValue=('TotalPrice', 'mean'),
    AvgUnitPrice=('UnitPrice', 'mean'),
    AvgQuantityPerLine=('Quantity', 'mean'),
    UniqueProducts=('StockCode', 'nunique'),
    FirstPurchaseDate=('InvoiceDate', 'min'),
    LastPurchaseDate=('InvoiceDate', 'max'),
    Country=('Country', 'first')
).reset_index()
customer_features['RecencyDays'] = (cutoff - customer_features['LastPurchaseDate']).dt.days
customer_features['TenureDays'] = (cutoff - customer_features['FirstPurchaseDate']).dt.days
month_diversity = pre_cutoff.assign(YearMonth=pre_cutoff['InvoiceDate'].dt.to_period('M')) \
    .groupby('CustomerID')['YearMonth'].nunique().rename('ActiveMonths')
customer_features = customer_features.merge(month_diversity, on='CustomerID')
pre_cutoff = pre_cutoff.copy()
pre_cutoff['IsWeekend'] = pre_cutoff['InvoiceDate'].dt.dayofweek >= 5
weekend_ratio = pre_cutoff.groupby('CustomerID')['IsWeekend'].mean().rename('WeekendPurchaseRatio')
customer_features = customer_features.merge(weekend_ratio, on='CustomerID')
customer_features['IsUK'] = (customer_features['Country'] == 'United Kingdom').astype(int)
customer_features.head()

,CustomerID,TotalSpend,PurchaseCount,AvgOrderValue,AvgUnitPrice,AvgQuantityPerLine,UniqueProducts,FirstPurchaseDate,LastPurchaseDate,Country,RecencyDays,TenureDays,ActiveMonths,WeekendPurchaseRatio,IsUK
0,12346.0,77183.60,1,77183.600000,1.040000,74215.000000,1,2011-01-18 10:01:00,2011-01-18 10:01:00,United Kingdom,225,225,1,0.0,1
1,12347.0,2790.86,5,22.506935,2.797661,12.822581,82,2010-12-07 14:57:00,2011-08-02 08:48:00,Iceland,29,267,5,0.0,0
2,12348.0,1487.24,3,53.115714,4.864643,75.857143,22,2010-12-16 19:09:00,2011-04-05 10:47:00,Finland,148,258,3,0.0,0
3,12350.0,334.40,1,19.670588,3.841176,11.588235,17,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway,210,210,1,0.0,0
4,12352.0,1561.81,5,41.100263,27.449474,6.684211,26,2011-02-16 12:33:00,2011-03-22 16:08:00,Norway,162,196,2,0.0,0


**Feature explanations (10 new features created):**

1. **TotalSpend** – Sum of all money spent before the cutoff. Directly reflects customer value (Monetary).
2. **PurchaseCount** – Number of distinct orders placed. Reflects how often the customer buys (Frequency).
3. **AvgOrderValue** – Average amount spent per order line. Shows typical spending pattern, not just total.
4. **AvgUnitPrice** – Average price per item bought. Signals whether the customer buys premium or budget items.
5. **AvgQuantityPerLine** – Average quantity bought per line item. Distinguishes bulk buyers from single-item buyers.
6. **UniqueProducts** – Number of distinct products purchased. Reflects breadth of interest in the catalog.
7. **RecencyDays** – Days since the customer's last purchase (relative to the cutoff). A core churn signal — the longer since their last visit, the more likely they've disengaged (Recency).
8. **TenureDays** – How long the customer has been active, from first purchase to the cutoff. Distinguishes new customers from long-term ones.
9. **ActiveMonths** – Number of distinct calendar months in which the customer made a purchase. Captures consistency versus a single one-off burst of buying.
10. **WeekendPurchaseRatio** – Fraction of a customer's purchases made on a Saturday/Sunday. A simple behavioral/timing signal extracted from the date column.
11. *(bonus)* **IsUK** – Simplified version of `Country`, since the vast majority of customers are UK-based; keeps the categorical signal without high-cardinality encoding.

## Step 6: Explain Every Feature

Each feature's meaning and rationale is documented directly above, next to where it was created (Step 5) — this keeps the explanation tied to the exact code that produces it, rather than separated into a disconnected list.

## Step 7: Check for Leakage

**Leakage check performed:**
- Every single feature above was computed using only `pre_cutoff` data — nothing from `post_cutoff` (the window used to define the target) was used anywhere in feature creation.
- `CustomerID` itself is used only as a grouping key, never as a model input — it carries no real predictive signal and including it as a raw feature could let the model "memorize" specific customers.
- `Description` and `StockCode` were **not** aggregated into features directly (e.g., no "most recent product bought") because product-level identifiers this close to the cutoff, combined with weak signal, risk overfitting to specific SKUs rather than genuine behavior. They were used only to compute a safe, aggregate signal (`UniqueProducts` — a count, not an identity).

**Result:** no feature here could "know" whether the customer purchased again after the cutoff — the target remains genuinely unseen at feature-creation time.

In [9]:
final_df = customer_features.merge(target_df, on='CustomerID')
print("Final dataset shape:", final_df.shape)
final_df.head()

Final dataset shape: (3317, 16)


,CustomerID,TotalSpend,PurchaseCount,AvgOrderValue,AvgUnitPrice,AvgQuantityPerLine,UniqueProducts,FirstPurchaseDate,LastPurchaseDate,Country,RecencyDays,TenureDays,ActiveMonths,WeekendPurchaseRatio,IsUK,WillPurchaseAgain
0,12346.0,77183.60,1,77183.600000,1.040000,74215.000000,1,2011-01-18 10:01:00,2011-01-18 10:01:00,United Kingdom,225,225,1,0.0,1,0
1,12347.0,2790.86,5,22.506935,2.797661,12.822581,82,2010-12-07 14:57:00,2011-08-02 08:48:00,Iceland,29,267,5,0.0,0,1
2,12348.0,1487.24,3,53.115714,4.864643,75.857143,22,2010-12-16 19:09:00,2011-04-05 10:47:00,Finland,148,258,3,0.0,0,1
3,12350.0,334.40,1,19.670588,3.841176,11.588235,17,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway,210,210,1,0.0,0,0
4,12352.0,1561.81,5,41.100263,27.449474,6.684211,26,2011-02-16 12:33:00,2011-03-22 16:08:00,Norway,162,196,2,0.0,0,1


## Step 8: Remove Irrelevant Features

Before modeling, drop columns that are identifiers, redundant, or not usable as numeric model inputs:

- `CustomerID` — identifier, not predictive
- `FirstPurchaseDate`, `LastPurchaseDate` — raw dates already converted into `TenureDays` / `RecencyDays`; keeping the raw dates would be redundant (and non-numeric)
- `Country` — replaced by the simpler `IsUK` flag to avoid a high-cardinality categorical column

In [10]:
model_df = final_df.drop(columns=['CustomerID', 'FirstPurchaseDate', 'LastPurchaseDate', 'Country'])
model_df.head()

,TotalSpend,PurchaseCount,AvgOrderValue,AvgUnitPrice,AvgQuantityPerLine,UniqueProducts,RecencyDays,TenureDays,ActiveMonths,WeekendPurchaseRatio,IsUK,WillPurchaseAgain
0,77183.60,1,77183.600000,1.040000,74215.000000,1,225,225,1,0.0,1,0
1,2790.86,5,22.506935,2.797661,12.822581,82,29,267,5,0.0,0,1
2,1487.24,3,53.115714,4.864643,75.857143,22,148,258,3,0.0,0,1
3,334.40,1,19.670588,3.841176,11.588235,17,210,210,1,0.0,0,0
4,1561.81,5,41.100263,27.449474,6.684211,26,162,196,2,0.0,0,1


## Step 9: Perform Feature Selection

With the irrelevant columns removed, we now check which of the *remaining* engineered features actually carry predictive signal, using two simple techniques:

1. **Correlation with the target** (quick, simple check for linear relationships)
2. **Feature importance from a Random Forest** (captures non-linear relationships too)

In [11]:
correlations = model_df.corr(numeric_only=True)['WillPurchaseAgain'].sort_values(ascending=False)
correlations

WillPurchaseAgain       1.000000
ActiveMonths            0.381364
UniqueProducts          0.259557
PurchaseCount           0.245421
TenureDays              0.137882
TotalSpend              0.122712
WeekendPurchaseRatio    0.001325
IsUK                   -0.006110
AvgQuantityPerLine     -0.022651
AvgOrderValue          -0.026801
AvgUnitPrice           -0.043141
RecencyDays            -0.308389
Name: WillPurchaseAgain, dtype: float64

In [12]:
from sklearn.ensemble import RandomForestClassifier
X = model_df.drop(columns=['WillPurchaseAgain'])
y = model_df['WillPurchaseAgain']
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importance

TotalSpend              0.144664
RecencyDays             0.128057
AvgUnitPrice            0.125672
UniqueProducts          0.114111
AvgOrderValue           0.114075
AvgQuantityPerLine      0.109934
TenureDays              0.100620
ActiveMonths            0.075380
PurchaseCount           0.054849
WeekendPurchaseRatio    0.025325
IsUK                    0.007314
dtype: float64

**Interpretation:** `RecencyDays`, `TenureDays`, and `PurchaseCount` come out as the strongest signals in both checks — which makes intuitive sense for a churn/retention problem. Weaker features like `WeekendPurchaseRatio` or `IsUK` could be dropped in a stricter final model, but are kept here since they add a small amount of independent signal without causing leakage or redundancy.

## Step 10: Compare the Dataset Before and After Feature Engineering

| | Before | After |
|---|---|---|
| **Grain (1 row =)** | 1 transaction line item | 1 customer |
| **Rows** | ~397,000 (cleaned) | ~3,300 customers |
| **Columns** | 8 raw columns, no target | 11 engineered features + 1 target |
| **Usable for ML?** | No — no target, wrong grain, raw text/IDs | Yes — numeric, customer-level, leakage-checked |
| **Signal captured** | Raw facts about individual purchases | Recency, frequency, monetary value, tenure, diversity, timing behavior |


In [13]:
print("BEFORE feature engineering:")
print(df[['InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country']].head(3))
print("\nAFTER feature engineering:")
model_df.head(3)

BEFORE feature engineering:
  InvoiceNo StockCode                         Description  Quantity  \
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                 WHITE METAL LANTERN         6   
2    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  

AFTER feature engineering:


,TotalSpend,PurchaseCount,AvgOrderValue,AvgUnitPrice,AvgQuantityPerLine,UniqueProducts,RecencyDays,TenureDays,ActiveMonths,WeekendPurchaseRatio,IsUK,WillPurchaseAgain
0,77183.60,1,77183.600000,1.040000,74215.000000,1,225,225,1,0.0,1,0
1,2790.86,5,22.506935,2.797661,12.822581,82,29,267,5,0.0,0,1
2,1487.24,3,53.115714,4.864643,75.857143,22,148,258,3,0.0,0,1
